In [23]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [24]:
sim = BasicSimulator()   # simulator backend used throughout

N               = 100   # number of qubits Alice transmits
CHECK_FRACTION  = 0.5   # fraction of sifted key sacrificed for error checking
ERROR_THRESHOLD = 0.1   # abort if error rate exceeds 10%

In [25]:
def quantum_random_bits(n):
    bits = []
    while len(bits) < n:
        size = min(20, n - len(bits))
        qc = QuantumCircuit(size, size)
        for i in range(size):
            qc.h(i)                           # |0> --> (|0>+|1>) / sqrt(2)
        qc.measure(range(size), range(size))  # collapses to 0 or 1 at random
        job = sim.run(transpile(qc, sim), shots=1, memory=True)
        bits_str = job.result().get_memory()[0]
        bits.extend([int(b) for b in reversed(bits_str)])
    return bits[:n]

In [26]:
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)      # |0> --> |1>
    if basis == 1:
        qc.h(0)      # rotate to X-basis
    return qc

def measure_qubit(qubit_qc, basis):
    qc = qubit_qc.copy()
    if basis == 1:
        qc.h(0)      # rotate X-basis --> Z-basis before measuring
    qc.measure(0, 0)
    job = sim.run(transpile(qc, sim), shots=1, memory=True)
    return int(job.result().get_memory()[0])

In [27]:
# ── ALICE ──────────────────────────────────────────────────────────────────

alice_bits  = quantum_random_bits(N)  # Alice's secret key bits
alice_bases = quantum_random_bits(N)  # 0 = Z-basis, 1 = X-basis

# Alice encodes each bit and sends — intercepted by Eve on the quantum channel
channel_qubits = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print("[ALICE] Key bits (first 20) :", alice_bits[:20])
print("[ALICE] Bases   (first 20) :", alice_bases[:20])
print(f"[ALICE] {N} qubits prepared and sent.")

[ALICE] Key bits (first 20) : [1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0]
[ALICE] Bases   (first 20) : [1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1]
[ALICE] 100 qubits prepared and sent.


In [28]:
# ── EVE (intercept-resend attack) ──────────────────────────────────────────

eve_bases        = quantum_random_bits(N)  # Eve randomly guesses a basis for each qubit
eve_results      = []                       # Eve's measurement outcomes
forwarded_qubits = []                       # Qubits Eve re-prepares and sends to Bob

for i in range(N):
    # Step 1: Eve intercepts and measures in her chosen basis
    result = measure_qubit(channel_qubits[i], eve_bases[i])
    eve_results.append(result)

    # Step 2: Eve re-prepares a qubit from her result and forwards it to Bob
    # If eve_bases[i] == alice_bases[i]: forwarded qubit is correct (no disturbance)
    # If eve_bases[i] != alice_bases[i]: forwarded qubit is disturbed (error introduced)
    forwarded_qubits.append(encode_qubit(result, eve_bases[i]))

eve_correct = sum(eve_bases[i] == alice_bases[i] for i in range(N))
print("[EVE] Bases chosen (first 20):", eve_bases[:20])
print(f"[EVE] Eve's basis matched Alice's {eve_correct}/{N} times ({eve_correct/N*100:.0f}%)")
print(f"[EVE] Alice and Bob don't know about the interception yet.")

[EVE] Bases chosen (first 20): [0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1]
[EVE] Eve's basis matched Alice's 52/100 times (52%)
[EVE] Alice and Bob don't know about the interception yet.


In [29]:
# ── BOB ────────────────────────────────────────────────────────────────────

bob_bases   = quantum_random_bits(N)  # Bob's random measurement bases
bob_results = [measure_qubit(forwarded_qubits[i], bob_bases[i]) for i in range(N)]

print("[BOB] Bases   (first 20):", bob_bases[:20])
print("[BOB] Results (first 20):", bob_results[:20])

[BOB] Bases   (first 20): [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1]
[BOB] Results (first 20): [1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0]


In [30]:
# ── SIFTING ────────────────────────────────────────────────────────────────

sifted_alice = []
sifted_bob   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:      # same basis → keep this bit
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Transmitted   : {N} qubits")
print(f"After sifting : {len(sifted_alice)} bits  (~{len(sifted_alice)/N*100:.0f}%, expected ~50%)")
print(f"\nSifted key (Alice) : {sifted_alice[:20]} ...")
print(f"Sifted key (Bob)   : {sifted_bob[:20]} ...")

Transmitted   : 100 qubits
After sifting : 48 bits  (~48%, expected ~50%)

Sifted key (Alice) : [1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1] ...
Sifted key (Bob)   : [0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0] ...


In [31]:
# ── ERROR CHECKING ─────────────────────────────────────────────────────────

n_sifted = len(sifted_alice)
n_check  = math.ceil(n_sifted * CHECK_FRACTION)

# Select check positions using quantum randomness:
# assign each sifted position a random 4-bit priority, sort, take the lowest n_check.
rand_bits  = quantum_random_bits(n_sifted * 4)
priorities = [sum(rand_bits[i*4 + b] << b for b in range(4)) for i in range(n_sifted)]
all_pos    = sorted(range(n_sifted), key=lambda i: priorities[i])
check_idx  = sorted(all_pos[:n_check])
key_idx    = [i for i in range(n_sifted) if i not in check_idx]

check_alice = [sifted_alice[i] for i in check_idx]
check_bob   = [sifted_bob[i]   for i in check_idx]

errors     = sum(a != b for a, b in zip(check_alice, check_bob))
error_rate = errors / n_check

print(f"Sifted key length       : {n_sifted}")
print(f"Bits used for check     : {n_check}")
print(f"Errors found            : {errors}")
print(f"Error rate              : {error_rate*100:.1f}%  (threshold: {ERROR_THRESHOLD*100:.0f}%)")
print(f"(Theoretical with full intercept-resend: ~25%)")
print()

if error_rate > ERROR_THRESHOLD:
    print("ATTACK DETECTED! Error rate exceeds threshold — protocol aborted.")
    print("Eve's measurements disturbed the qubits and left a detectable fingerprint.")
else:
    print("Error rate below threshold. Attack not detected this run (try more qubits).")

Sifted key length       : 48
Bits used for check     : 24
Errors found            : 8
Error rate              : 33.3%  (threshold: 10%)
(Theoretical with full intercept-resend: ~25%)

ATTACK DETECTED! Error rate exceeds threshold — protocol aborted.
Eve's measurements disturbed the qubits and left a detectable fingerprint.


In [32]:
# ── KEY COMPARISON (diagnostic only) ───────────────────────────────────────

final_key_alice = [sifted_alice[i] for i in key_idx]
final_key_bob   = [sifted_bob[i]   for i in key_idx]

key_errors     = sum(a != b for a, b in zip(final_key_alice, final_key_bob))
key_error_rate = key_errors / len(final_key_alice) if final_key_alice else 0

print(f"Remaining key length : {len(final_key_alice)} bits")
print(f"Key errors           : {key_errors} ({key_error_rate*100:.1f}%)")
print()
print(f"Alice's key : {final_key_alice}")
print(f"Bob's key   : {final_key_bob}")
print()

if key_error_rate == 0:
    print("Keys happen to match (Eve got lucky this run — try increasing N).")
else:
    print("Keys differ — Eve's attack corrupted the key.")

Remaining key length : 24 bits
Key errors           : 7 (29.2%)

Alice's key : [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
Bob's key   : [0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0]

Keys differ — Eve's attack corrupted the key.
